In [1]:
from src.improved_model import SimpleCNN
import torch.optim as optim
import matplotlib.pyplot as plt

from src.load_and_save import save_model
from src.model import SimpleNN
from src.pruning import get_intermediate_outputs_as_numpy
from src.training import get_accuracy
from src.utils import device
from settings import settings
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import torch.nn as nn
import torch

from src.pruning import is_all_layers_separated
import numpy as np


from src.load_and_save import load_model

cnn_model = SimpleCNN()
cnn_model.load_state_dict(torch.load(settings.models_path / 'convnet.pth'))
cnn_model.to(device)


C:\Users\frrit\AppData\Local\Temp\ipykernel_34184\1988482374.py:23: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  cnn_model.load_state_dict(torch.load(settings.models_path /

SimpleCNN(
  (layer1): Conv2d(1, 8, kernel_size=(5, 5), stride=(5, 5), padding=(1, 1))
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (layer2): Linear(in_features=288, out_features=54, bias=True)
  (layer3): Linear(in_features=54, out_features=10, bias=True)
)

In [2]:
# Define the number of classes in MNIST (digits 0-9)
num_classes = 10

# Define the transformation to apply to the images
transform = transforms.Compose([
    transforms.ToTensor(),  # Convert images to PyTorch tensors
])

# Custom transform to one-hot encode the labels
class OneHotEncode:
    def __init__(self, num_classes):
        self.num_classes = num_classes

    def __call__(self, label):
        return torch.eye(self.num_classes)[label]

# Load the full training dataset
full_train_dataset = datasets.MNIST(
    root=settings.data_path,
    train=True,
    download=True,
    transform=transform,
    target_transform=OneHotEncode(num_classes)
)

# Split the full training dataset into training and validation datasets
train_size = int(0.8 * len(full_train_dataset))  # 80% for training
val_size = len(full_train_dataset) - train_size  # 20% for validation
train_dataset, val_dataset = random_split(full_train_dataset, [train_size, val_size])

# Load the test dataset
test_dataset = datasets.MNIST(
    root=settings.data_path,
    train=False,
    download=True,
    transform=transform,
    target_transform=OneHotEncode(num_classes)
)

# Create DataLoaders for training, validation, and test sets
train_dataloader = DataLoader(train_dataset, batch_size=512, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=512, shuffle=False)
test_dataloader = DataLoader(test_dataset, batch_size=5000, shuffle=False)

In [3]:
test_data, _  = next(iter(test_dataloader))
test_data = test_data.to(device)
cnn_model.eval_mode()


In [4]:
input_binary = cnn_model.first_layer(test_data).detach().cpu().numpy().astype(np.int8)
output_binary = cnn_model.second_layer(cnn_model.first_layer(test_data)).detach().cpu().numpy().astype(np.int8)
# input_binary.s
print(input_binary.shape, output_binary.shape)
print(cnn_model.layer2.weight.shape)

weights = cnn_model.layer2.weight.T.detach().cpu().numpy()
bias = cnn_model.layer2.bias.T.detach().cpu().numpy()


(5000, 288) (5000, 54)
torch.Size([54, 288])


C:\Users\frrit\AppData\Local\Temp\ipykernel_34184\3880587837.py:8: UserWarning: The use of `x.T` on tensors of dimension other than 2 to reverse their shape is deprecated and it will throw an error in a future release. Consider `x.mT` to transpose batches of matrices or `x.permute(*torch.arange(x.ndim - 1, -1, -1))` to reverse the dimensions of a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\TensorShape.cpp:3701.)
  bias = cnn_model.layer2.bias.T.detach().cpu().numpy()


In [28]:
from collections import Counter


select_row: int = 0
X = input_binary
Y = output_binary[:, select_row]
W = weights[:, select_row]

# Remove columns where .99 fraction of values are identical:
threshold = .999
def most_common_fraction(column):
    counts = Counter(column)
    most_common_count = counts.most_common(1)[0][1]  # Frequency of the most common value
    return most_common_count / len(column)

# Apply to each column
fractions = np.apply_along_axis(most_common_fraction, axis=0, arr=X)

# Identify columns to keep
columns_to_keep = fractions < threshold
print(f"Columns rejected: {X.shape[1] - sum(columns_to_keep)}")

# Filter the columns
X = X[:, columns_to_keep]
W = W[columns_to_keep]

# Keep only unique rows:
X, indices = np.unique(X, axis=0, return_index=True)
Y = Y[indices]

y_rec = X @ W
assert np.min(y_rec[Y==1]) > np.max(y_rec[Y==-1])
B = - (np.min(y_rec[Y==1]) + np.max(y_rec[Y==-1])) / 2

assert all(np.sign(X @ W + B) == Y)
# X @ W + B == Y

Columns rejected: 65


In [34]:
np.min(X @ W + B)

-48.966194

In [68]:
# assert np.sign(X @ W + B) == Y
Y_0 = Y[Y==-1]
X_0 = X[Y==-1,:]
Y_1 = Y[Y==1]
X_1 = X[Y==1,:]

import numpy as np


X_1 = np.sign(X_1)
# Compute distances to the hyperplane
distances = np.abs(X_1 @ W + B) / np.linalg.norm(W)

# Find the closest points (n = dimensionality of the space)
n = X.shape[1] + 100# Number of dimensions
closest_indices = np.argsort(distances)[:n]
closest_points = X_1[closest_indices]
# closest_points = X_1[distances < .25, :]

# Fit a new hyperplane through the selected points
# Solve for W_new and B_new such that W_new @ closest_points[i] + B_new = 0
A = np.hstack([closest_points, np.ones((len(closest_points), 1))])  # Add bias term
target = np.ones(A.shape[0])

new_plane = np.linalg.pinv(A)@target

# Extract W_new and B_new from the result
W_new = new_plane[:-1]
B_new = new_plane[-1]

print(f"New hyperplane normal vector (W_new): {W_new}")
print(f"New hyperplane bias (B_new): {B_new}")


New hyperplane normal vector (W_new): [ 5.20417043e-17 -2.15105711e-16  2.01227923e-16 -2.22044605e-16
 -2.30718222e-16 -4.30211422e-16  1.97758476e-16 -6.24500451e-17
 -1.52655666e-16  8.29197822e-16  4.37926878e-02  2.01227923e-16
  1.38777878e-16 -1.17961196e-16 -7.28583860e-17 -9.24065890e-03
  4.37926878e-02 -3.81639165e-17 -3.46944695e-18  1.73472348e-16
 -2.91433544e-16  9.24065890e-03  1.30104261e-16 -1.05818132e-16
 -8.15320034e-17  3.03576608e-16  6.02651667e-03 -4.37926878e-02
  2.90132501e-16  2.08166817e-16  1.02348685e-16 -3.66026653e-16
  7.35522754e-16  2.11636264e-16  3.46944695e-16  1.73472348e-16
  8.67361738e-17  1.94289029e-16  1.02695630e-15 -4.37926878e-02
  5.68989300e-16 -3.03576608e-17 -2.90566182e-16 -3.01841885e-16
  1.95590072e-16 -4.37926878e-02  1.38777878e-17  1.38777878e-17
 -2.46330734e-16 -1.56125113e-16 -1.96866211e-02  7.35522754e-16
  1.50920942e-16  1.52655666e-16 -3.19189120e-16 -1.04459622e-02
 -1.38777878e-16 -4.37926878e-02 -4.16333634e-17  1.

In [69]:
print( np.mean(np.sign(X@W_new + B_new - 1) == Y))


0.7091018444266239
